<a href="https://colab.research.google.com/github/prishaarorain-collab/Agentic-AI-in-Customer-Service-and-Sales/blob/main/Agentic_AI_in_Customer_Service_and_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Service and Sales Chatbot
**Multi-agent chatbot using LangGraph + Google Gemini API**

Based on starter code: `code_06_XX Multi-agent chatbots with routing`

**Agents:**
- **Product Agent** — laptop features & pricing
- **Orders Agent** — order status, quantity updates & refund policy
- **Small Talk Agent** — greetings and goodbyes
- **Router Agent** — classifies intent and routes to the right agent

**Memory:** LangGraph `MemorySaver` maintains conversation state across 3+ turns

## Setup API Key


In [1]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key loaded.")

API key loaded.


## Install Dependencies

In [2]:
!pip install -q langchain-google-genai langgraph langchain langchain-community pandas
print("Done installing.")

Done installing.


## Cell 1 — Setup Google Gemini
Replaces `AzureChatOpenAI` from starter code

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Setup the LLM — replaces AzureChatOpenAI from starter code
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)
print("Google Gemini model ready.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Google Gemini model ready.


## Tools
Product and order tools using CSV data

In [4]:
import pandas as pd
from langchain_core.tools import tool

# Load laptop pricing CSV
product_pricing_df = pd.read_csv("data/Laptop pricing.csv")
print(product_pricing_df)

@tool
def get_laptop_price(laptop_name: str) -> str:
    """Returns the price and shipping days for a laptop given its name."""
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains("^" + laptop_name, case=False)
    ]
    if len(match_df) == 0:
        return f"No laptop found matching '{laptop_name}'."
    return str(match_df.iloc[0].to_dict())

@tool
def get_laptop_features(laptop_name: str) -> str:
    """Returns available product info for a laptop given its name."""
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains("^" + laptop_name, case=False)
    ]
    if len(match_df) == 0:
        return f"No laptop found matching '{laptop_name}'."
    return str(match_df.iloc[0].to_dict())

# Load orders CSV
product_orders_df = pd.read_csv("data/Laptop Orders.csv")
print(product_orders_df)

@tool
def get_order_details(order_id: str) -> str:
    """Returns details about a laptop order given its order ID."""
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return f"No order found with ID '{order_id}'."
    return str(match_df.iloc[0].to_dict())

@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """Updates the quantity of laptops for a given order ID."""
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return f"No order found with ID '{order_id}'."
    product_orders_df.loc[product_orders_df["Order ID"] == order_id, "Quantity Ordered"] = new_quantity
    return f"Order {order_id} quantity updated to {new_quantity}."

@tool
def get_refund_policy(topic: str) -> str:
    """Returns the refund/return policy. Topic: general, damaged, wrong_item, or timeline."""
    policies = {
        "general": "Laptops can be returned within 30 days of delivery in original condition.",
        "damaged": "Report damaged items within 7 days with photos for a full refund or replacement.",
        "wrong_item": "Wrong items will be collected for free and replaced immediately.",
        "timeline": "Refunds are processed within 5-7 business days after item is received."
    }
    for key in policies:
        if key in topic.lower():
            return policies[key]
    return policies["general"]

print("All tools ready.")

            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2
   Order ID Product Ordered  Quantity Ordered Delivery Date
0  ORD-8276   SpectraBook S                 3    2024-10-16
1  ORD-6948    OmegaPro G17                 3    2024-10-25
2  ORD-7311   NanoEdge Flex                 2    2024-10-19
3  ORD-4633    OmegaPro G17                 2    2024-10-15
4  ORD-2050      GammaAir X                 2    2024-10-26
All tools ready.


## Base Agent Class
Reusable agent class using `StateGraph` — same pattern as `code_04`

In [5]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from typing import TypedDict, Annotated
import operator

# Reusable agent state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# Reusable agent class — same pattern as OrdersAgent in code_04
class CustomerServiceAgent:

    def __init__(self, model, tools, system_prompt, debug=False):
        self.system_prompt = system_prompt
        self.debug = debug

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("tools", self.call_tools)
        graph.add_conditional_edges("llm", self.is_tool_call, {True: "tools", False: END})
        graph.add_edge("tools", "llm")
        graph.set_entry_point("llm")

        self.memory = MemorySaver()
        self.graph = graph.compile(checkpointer=self.memory)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_llm(self, state: AgentState):
        messages = [SystemMessage(content=self.system_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        return {"messages": [result]}

    def is_tool_call(self, state: AgentState):
        return len(state["messages"][-1].tool_calls) > 0

    def call_tools(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for tc in tool_calls:
            result = self.tools[tc["name"]].invoke(tc["args"]) if tc["name"] in self.tools else "Tool not found."
            results.append(ToolMessage(tool_call_id=tc["id"], name=tc["name"], content=str(result)))
        return {"messages": results}

print("Base agent class ready.")

Base agent class ready.


## Product & Orders Agents
Two specialist agents, each with their own system prompt

In [6]:
# Product Agent — handles laptop features and pricing questions
product_agent = CustomerServiceAgent(
    model=model,
    tools=[get_laptop_price, get_laptop_features],
    system_prompt="""
    You are a professional chatbot that answers questions about laptops sold by your company.
    Use the available tools to answer questions about laptop pricing and features.
    Be helpful and professional in all responses.
    """
)

# Orders Agent — handles order status, updates, and refund policy
orders_agent = CustomerServiceAgent(
    model=model,
    tools=[get_order_details, update_quantity, get_refund_policy],
    system_prompt="""
    You are a professional chatbot that manages laptop orders for our company.
    Use tools to retrieve order details, update quantities, and explain refund policy.
    Do NOT reveal information about other orders than the one requested.
    """
)

print("Product Agent ready.")
print("Orders Agent ready.")

Product Agent ready.
Orders Agent ready.


## Router Agent
From `code_06` — routes each message to the correct specialist agent

In [7]:
import functools

def agent_node(state, agent, name, config):
    """Helper to invoke a sub-agent and return its response."""
    thread_id = config["metadata"]["thread_id"]
    result = agent.graph.invoke(state, {"configurable": {"thread_id": thread_id}})
    return {"messages": [AIMessage(result["messages"][-1].content)]}

product_node = functools.partial(agent_node, agent=product_agent, name="Product_Agent")
orders_node  = functools.partial(agent_node, agent=orders_agent,  name="Orders_Agent")

class RouterState(TypedDict):
    messages: Annotated[list, operator.add]

class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        self.system_prompt = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model = model
        self.debug = debug

        g = StateGraph(RouterState)
        g.add_node("Router",        self.classify)
        g.add_node("Product_Agent", product_node)
        g.add_node("Orders_Agent",  orders_node)
        g.add_node("Small_Talk",    self.smalltalk)

        g.add_conditional_edges("Router", self.find_route,
            {"PRODUCT": "Product_Agent", "ORDER": "Orders_Agent",
             "SMALLTALK": "Small_Talk", "END": END})

        g.add_edge("Product_Agent", END)
        g.add_edge("Orders_Agent",  END)
        g.add_edge("Small_Talk",    END)
        g.set_entry_point("Router")
        self.graph = g.compile()

    def classify(self, state: RouterState):
        """Router LLM: classify the intent."""
        messages = [SystemMessage(content=self.system_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        if self.debug: print(f"Classified: {result.content}")
        return {"messages": [result]}

    def smalltalk(self, state: RouterState):
        """Handle greetings and casual chat."""
        messages = [SystemMessage(content=self.smalltalk_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        if isinstance(result.content, list):
            result.content = " ".join([c if isinstance(c, str) else c.get("text", "") for c in result.content]).strip()
        return {"messages": [result]}

    def find_route(self, state: RouterState):
        """Return the destination node based on router output."""
        dest = state["messages"][-1].content.strip()
        if self.debug: print(f"Routing to: {dest}")
        return dest if dest in ["PRODUCT", "ORDER", "SMALLTALK"] else "END"


router_agent = RouterAgent(
    model=model,
    system_prompt="""
You are a Router. Analyze the user's message and respond with exactly one word:
SMALLTALK  - greetings, goodbyes, or casual chat
PRODUCT    - questions about laptop features, specs, or pricing
ORDER      - questions about orders, delivery, quantity, or refund policy
END        - anything else

Respond with ONLY that one word. Nothing else.
""",
    smalltalk_prompt="""
Respond professionally to greetings and goodbyes.
Let the user know you can help with laptop product questions and order management.
""",
    debug=False
)

print("Router Agent ready. Full multi-agent chatbot is online!")

Router Agent ready. Full multi-agent chatbot is online!


## Multi-Turn Conversation Demo
Shows all 3 agent types with MemorySaver keeping context

In [8]:
import uuid

user_inputs = [
    "Hello, how are you?",
    "Tell me about the SpectraBook S",
    "How much does it cost?",
    "Show me the details of order ORD-7311",
    "Can you add one more laptop to that order?",
    "What is your refund policy?",
    "Compare GammaAir X and AlphaBook Pro prices",
    "Goodbye!",
]

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("\n" + "="*60)
print("   CUSTOMER SERVICE CHATBOT — MULTI-AGENT DEMO")
print("="*60)

for user_input in user_inputs:
    print(f"\n{'─'*55}")
    print(f"USER  : {user_input}")
    response = router_agent.graph.invoke(
        {"messages": [HumanMessage(user_input)]}, config=config
    )
    print(f"AGENT : {response['messages'][-1].content}")

print("\n" + "="*60)
print("DEMO COMPLETE")
print("="*60)


   CUSTOMER SERVICE CHATBOT — MULTI-AGENT DEMO

───────────────────────────────────────────────────────
USER  : Hello, how are you?
AGENT : Hello! I'm doing well, thank you for asking.

I can help you with any laptop product questions you might have, or assist with order management. How can I help you today?

───────────────────────────────────────────────────────
USER  : Tell me about the SpectraBook S
AGENT : The SpectraBook S is a high-end laptop that costs $2499. It ships in 7 days. 


───────────────────────────────────────────────────────
USER  : How much does it cost?
AGENT : The SpectraBook S costs $2499.

───────────────────────────────────────────────────────
USER  : Show me the details of order ORD-7311
AGENT :  ID: ORD-7311
LAPTOP MODEL: P15v Gen 3
QUANTITY: 1
PRICE: $1,729.00
STATUS: Shipped
ESTIMATED DELIVERY: 2024-07-11

───────────────────────────────────────────────────────
USER  : Can you add one more laptop to that order?
AGENT :  ID: ORD-7311
LAPTOP MODEL: P15v Gen

## Cell 7 — Memory Test: 3-Turn Conversation
Shows MemorySaver retaining context across 3 turns

In [9]:
print("="*55)
print("MEMORY TEST — 3-Turn Conversation")
print("="*55)

memory_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

turns = [
    "What are the details of order ID ORD-6948?",
    "Can you increase the quantity of it to 5?",
    "Can you show me the updated details of the order?",
]

for turn in turns:
    print(f"\n{'─'*50}")
    print(f"USER  : {turn}")
    result = router_agent.graph.invoke(
        {"messages": [HumanMessage(turn)]}, config=memory_config
    )
    print(f"AGENT : {result['messages'][-1].content}")

print("\nMemorySaver successfully retained context across all 3 turns.")

MEMORY TEST — 3-Turn Conversation

──────────────────────────────────────────────────
USER  : What are the details of order ID ORD-6948?
AGENT : The details for Order ID ORD-6948 are:
Product Ordered: OmegaPro G17
Quantity Ordered: 3
Delivery Date: 2024-10-25

──────────────────────────────────────────────────
USER  : Can you increase the quantity of it to 5?
AGENT : The quantity for Order ID ORD-6948 has been updated to 5.

──────────────────────────────────────────────────
USER  : Can you show me the updated details of the order?
AGENT : The updated details for Order ID ORD-6948 are:
Product Ordered: OmegaPro G17
Quantity Ordered: 5
Delivery Date: 2024-10-25

MemorySaver successfully retained context across all 3 turns.
